# KYNTRA_01_OVERTAKE_EDA

**Purpose:** Exploratory Data Analysis for KYNTRA's verified 2026 overtake dataset.

> Do not run this notebook until the final same-lap event audit is approved.

This notebook must NOT:
- train any model
- tune thresholds
- inspect demo holdout outcomes for feature decisions
- merge the 2024 auxiliary dataset
- invent probabilities


In [ ]:
!pip -q install pyarrow duckdb

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import duckdb

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)


## Mount Google Drive


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


## Paths


In [ ]:
ROOT = Path("/content/drive/MyDrive/KYNTRA")
DATA = ROOT / "datasets"
FIGURES = ROOT / "figures"
EXPERIMENTS = ROOT / "experiments"

FIGURES.mkdir(parents=True, exist_ok=True)
EXPERIMENTS.mkdir(parents=True, exist_ok=True)

PRIMARY = DATA / "kyntra_overtake_dataset.parquet"

assert PRIMARY.exists(), f"Dataset not found: {PRIMARY}"


## Load dataset


In [ ]:
df = pd.read_parquet(PRIMARY)

print("shape:", df.shape)
display(df.head())


## Absolute integrity checks


In [ ]:
assert df["observation_id"].is_unique
assert set(df["split"].dropna().unique()).issubset({"TRAIN", "VALIDATION"})

print(df["split"].value_counts())
print("events:", df["event_id"].nunique())
print("battle sequences:", df["battle_sequence_id"].nunique())


## Separate TRAIN and VALIDATION


In [ ]:
train = df[df["split"] == "TRAIN"].copy()
val = df[df["split"] == "VALIDATION"].copy()

print("TRAIN:", train.shape)
print("VALIDATION:", val.shape)

assert set(train["event_id"]).isdisjoint(set(val["event_id"]))
assert set(train["battle_sequence_id"]).isdisjoint(set(val["battle_sequence_id"]))


## Target and censoring audit


In [ ]:
horizons = [1, 2, 3]
rows = []

for h in horizons:
    target = f"overtake_next_{h}_lap" if h == 1 else f"overtake_next_{h}_laps"
    cens = f"censored_{h}_lap" if h == 1 else f"censored_{h}_laps"

    for name, part in [("TRAIN", train), ("VALIDATION", val)]:
        valid = part[part[cens] != 1]
        rows.append({
            "split": name,
            "horizon": h,
            "rows": len(part),
            "censored": int(part[cens].fillna(0).sum()),
            "usable_rows": len(valid),
            "positives": int(valid[target].fillna(0).sum()),
            "positive_rate": float(valid[target].mean()),
        })

target_summary = pd.DataFrame(rows)
display(target_summary)


## Retention audit


In [ ]:
retention_rows = []

for h in horizons:
    retained = f"retained_position_{h}_lap" if h == 1 else f"retained_position_{h}_laps"
    repassed = f"repassed_within_{h}_lap" if h == 1 else f"repassed_within_{h}_laps"
    rcens = f"retention_censored_{h}_lap" if h == 1 else f"retention_censored_{h}_laps"

    for name, part in [("TRAIN", train), ("VALIDATION", val)]:
        eligible = part[part[[retained, repassed, rcens]].notna().any(axis=1)]
        retention_rows.append({
            "split": name,
            "horizon": h,
            "eligible": len(eligible),
            "retained": int(eligible[retained].fillna(0).sum()),
            "repassed": int(eligible[repassed].fillna(0).sum()),
            "censored": int(eligible[rcens].fillna(0).sum()),
        })

retention_summary = pd.DataFrame(retention_rows)
display(retention_summary)


## Missingness


In [ ]:
missing = (
    train.isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .rename("missing_pct")
    .to_frame()
)

display(missing.head(50))


## Event-level balance


In [ ]:
event_summary = (
    train.groupby(["event_id", "event_name"], dropna=False)
    .agg(
        observations=("observation_id", "size"),
        battle_sequences=("battle_sequence_id", "nunique"),
    )
    .reset_index()
)

display(event_summary.sort_values("observations", ascending=False))


## Battle sequence lengths


In [ ]:
seq_len = train.groupby("battle_sequence_id").size().rename("sequence_length")
display(seq_len.describe(percentiles=[.01, .1, .25, .5, .75, .9, .99]))

plt.figure(figsize=(9, 5))
plt.hist(seq_len, bins=40)
plt.title("TRAIN Battle Sequence Length")
plt.xlabel("Observations / laps in sequence")
plt.ylabel("Count")
plt.show()


## Continuous feature sanity


In [ ]:
continuous = [
    "distance_gap_m",
    "gap_seconds",
    "closing_rate",
    "recent_pace_delta_1lap",
    "recent_pace_delta_3laps",
    "sector1_delta",
    "sector2_delta",
    "sector3_delta",
    "speed_trap_delta",
    "tyre_age_delta",
    "laps_remaining",
    "consecutive_laps_following",
    "consecutive_laps_close",
    "distance_gap_mean_recent",
    "distance_gap_std_recent",
    "rear_distance_gap_m",
    "weather_air_temp",
    "weather_track_temp",
]

existing_continuous = [c for c in continuous if c in train.columns]

display(
    train[existing_continuous]
    .describe(percentiles=[.01, .05, .25, .5, .75, .95, .99])
    .T
)


## Categorical distributions


In [ ]:
categorical = [
    "attacker_compound",
    "defender_compound",
    "race_phase",
    "rear_threat_proxy",
    "track_status_parsed",
    "weather_rainfall",
]

for col in categorical:
    if col in train.columns:
        print("\n###", col)
        display(train[col].value_counts(dropna=False).head(20))


## PASS vs NO PASS — TRAIN only


In [ ]:
TARGET = "overtake_next_1_lap"
CENSOR = "censored_1_lap"

eda1 = train[train[CENSOR] != 1].copy()

group_summary = (
    eda1.groupby(TARGET)[existing_continuous]
    .median(numeric_only=True)
    .T
)

display(group_summary)


## PASS + RETAIN vs PASS + REPASS — signature KYNTRA analysis


In [ ]:
pass_rows = train[
    (train["overtake_next_1_lap"] == 1)
    & (train["retention_censored_1_lap"] != 1)
].copy()

def durability_class(row):
    if row["retained_position_1_lap"] == 1:
        return "PASS_RETAIN"
    if row["repassed_within_1_lap"] == 1:
        return "PASS_REPASS"
    return "OTHER"

pass_rows["durability_class"] = pass_rows.apply(durability_class, axis=1)

display(pass_rows["durability_class"].value_counts())

durability_summary = (
    pass_rows[pass_rows["durability_class"].isin(["PASS_RETAIN", "PASS_REPASS"])]
    .groupby("durability_class")[existing_continuous]
    .median(numeric_only=True)
    .T
)

display(durability_summary)


## Gap vs closing-rate visualization


In [ ]:
plot_df = eda1.dropna(subset=["gap_seconds", "closing_rate", TARGET]).copy()

plt.figure(figsize=(9, 6))
for label, grp in plot_df.groupby(TARGET):
    plt.scatter(
        grp["gap_seconds"],
        grp["closing_rate"],
        s=8,
        alpha=0.25,
        label=f"target={label}",
    )
plt.xlabel("Gap seconds")
plt.ylabel("Closing rate")
plt.title("TRAIN — Gap vs Closing Rate")
plt.legend()
plt.show()


## TRAIN vs VALIDATION shift check


In [ ]:
shift = []

for col in existing_continuous:
    shift.append({
        "feature": col,
        "train_median": train[col].median(),
        "validation_median": val[col].median(),
        "train_missing_pct": train[col].isna().mean() * 100,
        "validation_missing_pct": val[col].isna().mean() * 100,
    })

shift_df = pd.DataFrame(shift)
display(shift_df)


## Save EDA tables


In [ ]:
target_summary.to_csv(EXPERIMENTS / "eda_target_summary.csv", index=False)
retention_summary.to_csv(EXPERIMENTS / "eda_retention_summary.csv", index=False)
event_summary.to_csv(EXPERIMENTS / "eda_event_summary.csv", index=False)
missing.to_csv(EXPERIMENTS / "eda_missingness_train.csv")
shift_df.to_csv(EXPERIMENTS / "eda_train_validation_shift.csv", index=False)

print("EDA tables saved.")


# STOP HERE

Before starting `KYNTRA_02_BASELINE.ipynb`, review:

1. usable observations after censoring
2. 1/2/3-lap positive rates
3. class imbalance
4. event concentration
5. missingness
6. suspicious outliers
7. battle-sequence lengths
8. PASS_RETAIN vs PASS_REPASS sample sizes
9. TRAIN vs VALIDATION shift
10. feature removals needed before modeling
